# Use PMML and Batch Deployments to predict iris species with `ibm-watsonx-ai`

This notebook contains steps from storing sample PMML model to starting scoring new data using batch deployment. 

Some familiarity with python is helpful. This notebook uses Python 3.12.

You will use a **Iris** data set, which details measurements of iris perianth. Use the details of this data set to predict iris species.

## Learning goals

The learning goals of this notebook are:

-  Working with the watsonx.ai Runtime instance
-  Batch deployment of PMML model
-  Scoring of deployed model


## Contents

This notebook contains the following parts:

1. [Set up the environment](#1.-Set-up-the-environment)
2. [Upload model](#2.-Upload-model)
3. [Create batch deployment](#3.-Create-batch-deployment)
4. [Scoring](#4.-Scoring)
5. [Cleanup](#5.-Cleanup)
6. [Summary and next steps](#6.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

### Connection to watsonx.ai Runtime

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform `api_key` and instance `location`.

You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve platform API Key and instance location.

API Key can be generated in the following way:
```
ibmcloud login
ibmcloud iam api-key-create API_KEY_NAME
```

In result, get the value of `api_key` from the output.


Location of your watsonx.ai Runtime instance can be retrieved in the following way:
```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```

In result, get the value of `location` from the output.

**Tip**: Your `Cloud API key` can be generated by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.

You can also get service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, then copy the created key and paste it below.

**Action**: Enter your `url` and `api_key` in the following cell.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

In [3]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have space already created, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=cpdaas) to create one.

- Click New Deployment Space
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press Create
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below

In [4]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai Runtime, you need to set **space** which you will be using.

In [5]:
client.set.default_space(space_id)

'SUCCESS'

<a id="2.-Upload-model"></a>
## 2. Upload model

In this section you will learn how to upload the model to the Cloud.

**Action**: Download sample PMML model from git project using wget.

In [6]:
import os

from wget import download

sample_dir = "pmml_sample_model"
if not os.path.isdir(sample_dir):
    os.mkdir(sample_dir)

filename = os.path.join(sample_dir, "iris_chaid.xml")
if not os.path.isfile(filename):
    filename = download(
        "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/models/pmml/iris-species/model/iris_chaid.xml",
        out=sample_dir,
    )

Store downloaded file in watsonx.ai Runtime repository.

In [7]:
sw_spec_id = client.software_specifications.get_id_by_name("pmml-3.0_4.3")

meta_props = {
    client.repository.ModelMetaNames.NAME: "pmmlmodel",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: sw_spec_id,
    client.repository.ModelMetaNames.TYPE: "pmml_4.2.1",
}

In [8]:
published_model = client.repository.store_model(model=filename, meta_props=meta_props)

**Note:** You can see that model is successfully stored in watsonx.ai Runtime Service.

In [9]:
client.repository.list_models()

,ID,NAME,CREATED,TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,642539e8-fe58-4102-a5c4-3d774e3aa3d8,pmmlmodel,2026-02-17T13:57:55Z,pmml_4.2.1,supported,


<a id="3.-Create-batch-deployment"></a>
## 3. Create batch deployment

You can use command below to create batch deployment for stored model.

In [10]:
model_id = client.repository.get_model_id(published_model)
deployment = client.deployments.create(
    artifact_id=model_id,
    meta_props={
        client.deployments.ConfigurationMetaNames.NAME: "Sample PMML Batch deployment",
        client.deployments.ConfigurationMetaNames.BATCH: {},
        client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {
            "name": "S",
            "num_nodes": 1,
        },
    },
)



######################################################################################

Synchronous deployment creation for id: '642539e8-fe58-4102-a5c4-3d774e3aa3d8' started

######################################################################################


ready.


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='308ad8c8-4428-438a-a96e-aba42355de4c'
-----------------------------------------------------------------------------------------------




Batch deployment has been created.

You can retrieve now your deployment ID.

In [11]:
deployment_id = client.deployments.get_id(deployment)

You can also list all deployments in your space.

In [12]:
client.deployments.list()

,ID,NAME,STATE,CREATED,ARTIFACT_TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,308ad8c8-4428-438a-a96e-aba42355de4c,Sample PMML Batch deployment,ready,2026-02-17T13:58:05.661Z,model,supported,


If you want to get additional information on your deployment, you can do it as below.

In [13]:
client.deployments.get_details(deployment_id)

{'entity': {'asset': {'id': '642539e8-fe58-4102-a5c4-3d774e3aa3d8'},
  'batch': {},
  'chat_enabled': False,
  'custom': {},
  'deployed_asset_type': 'model',
  'hardware_spec': {'name': 'S', 'num_nodes': 1},
  'name': 'Sample PMML Batch deployment',
  'space_id': 'fb3d528a-bf16-460e-bcf6-06f05d8ba57c',
  'status': {'state': 'ready'}},
 'metadata': {'created_at': '2026-02-17T13:58:05.661Z',
  'id': '308ad8c8-4428-438a-a96e-aba42355de4c',
  'modified_at': '2026-02-17T13:58:05.661Z',
  'name': 'Sample PMML Batch deployment',
  'owner': 'IBMid-696000GJGB',
  'space_id': 'fb3d528a-bf16-460e-bcf6-06f05d8ba57c'}}

<a id="4.-Scoring"></a>
## 4. Scoring

You can send new scoring records to batch deployment using by creating job.

In [14]:
job_payload_ref = {
    client.deployments.ScoringMetaNames.INPUT_DATA: [
        {
            "fields": ["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width"],
            "values": [[5.1, 3.5, 1.4, 0.2]],
        }
    ]
}

job = client.deployments.create_job(deployment_id, meta_props=job_payload_ref)

Now, your job has been submitted to runtime.

You can retrieve now your job ID.

In [15]:
job_id = client.deployments.get_job_id(job)

You can also list all jobs in your space.

In [16]:
client.deployments.list_jobs()

,JOB-ID,STATE,CREATED,DEPLOYMENT-ID
0,67fa9821-3246-41b1-ad61-147a1309460b,queued,2026-02-17T13:58:16.738Z,308ad8c8-4428-438a-a96e-aba42355de4c


If you want to get additional information on your job, you can do it as below.

In [17]:
client.deployments.get_job_details(job_id)

{'entity': {'deployment': {'id': '308ad8c8-4428-438a-a96e-aba42355de4c'},
  'platform_job': {'job_id': 'ec82edd3-d89c-471a-a974-584904c0d0d3',
   'run_id': '67fa9821-3246-41b1-ad61-147a1309460b'},
  'scoring': {'input_data': [{'fields': ['Sepal.Length',
      'Sepal.Width',
      'Petal.Length',
      'Petal.Width'],
     'values': [[5.1, 3.5, 1.4, 0.2]]}],
   'status': {'completed_at': '', 'running_at': '', 'state': 'queued'}}},
 'metadata': {'created_at': '2026-02-17T13:58:16.738Z',
  'id': '67fa9821-3246-41b1-ad61-147a1309460b',
  'name': 'name_2a619dfe-41f1-414b-bca0-856154c2b3b4',
  'space_id': 'fb3d528a-bf16-460e-bcf6-06f05d8ba57c'}}

### Monitor job execution
Here you can check status of your batch scoring.

In [18]:
import time

elapsed_time = 0
while (
    client.deployments.get_job_status(job_id).get("state") != "completed"
    and elapsed_time < 300
):
    print(f" Current state: {client.deployments.get_job_status(job_id).get('state')}")
    elapsed_time += 10
    time.sleep(10)

if client.deployments.get_job_status(job_id).get("state") == "completed":
    print(f" Current state: {client.deployments.get_job_status(job_id).get('state')}")
    job_details_do = client.deployments.get_job_details(job_id)
    print(job_details_do)
else:
    print("Job hasn't completed successfully in 5 minutes.")

 Current state: queued
 Current state: queued
 Current state: completed
{'entity': {'deployment': {'id': '308ad8c8-4428-438a-a96e-aba42355de4c'}, 'platform_job': {'job_id': 'ec82edd3-d89c-471a-a974-584904c0d0d3', 'run_id': '67fa9821-3246-41b1-ad61-147a1309460b'}, 'scoring': {'input_data': [{'fields': ['Sepal.Length', 'Sepal.Width', 'Petal.Length', 'Petal.Width'], 'values': [[5.1, 3.5, 1.4, 0.2]]}], 'predictions': [{'fields': ['$R-Species', '$RC-Species', '$RP-Species', '$RP-setosa', '$RP-versicolor', '$RP-virginica', '$RI-Species'], 'values': [['setosa', 1.0, 1.0, 1.0, 0.0, 0.0, '1']]}], 'status': {'completed_at': '2026-02-17T13:58:37.000Z', 'running_at': '2026-02-17T13:58:37.000Z', 'state': 'completed'}}}, 'metadata': {'created_at': '2026-02-17T13:58:16.738Z', 'id': '67fa9821-3246-41b1-ad61-147a1309460b', 'modified_at': '2026-02-17T13:58:37.723Z', 'name': 'name_2a619dfe-41f1-414b-bca0-856154c2b3b4', 'space_id': 'fb3d528a-bf16-460e-bcf6-06f05d8ba57c'}}


Get scored data

In [19]:
import json

print(json.dumps(client.deployments.get_job_details(job_id), indent=2))

{
  "entity": {
    "deployment": {
      "id": "308ad8c8-4428-438a-a96e-aba42355de4c"
    },
    "platform_job": {
      "job_id": "ec82edd3-d89c-471a-a974-584904c0d0d3",
      "run_id": "67fa9821-3246-41b1-ad61-147a1309460b"
    },
    "scoring": {
      "input_data": [
        {
          "fields": [
            "Sepal.Length",
            "Sepal.Width",
            "Petal.Length",
            "Petal.Width"
          ],
          "values": [
            [
              5.1,
              3.5,
              1.4,
              0.2
            ]
          ]
        }
      ],
      "predictions": [
        {
          "fields": [
            "$R-Species",
            "$RC-Species",
            "$RP-Species",
            "$RP-setosa",
            "$RP-versicolor",
            "$RP-virginica",
            "$RI-Species"
          ],
          "values": [
            [
              "setosa",
              1.0,
              1.0,
              1.0,
              0.0,
              0.0,
   

As we can see this is Iris Setosa flower.

<a id="5.-Cleanup"></a>
## 5. Cleanup

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="6.-Summary-and-next-steps"></a>
## 6. Summary and next steps

You successfully completed this notebook! You learned how to use watsonx.ai Runtime for PMML model deployment and scoring. Check out our _[Online Documentation](https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=wx)_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Jan Sołtysik (Former)**, Software Engineer at IBM watsonx.ai

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2020-2026 IBM. This notebook and its source code are released under the terms of the MIT License.